## load libs

In [1]:
import cpufeature as cpufeature
import dcimg as dcimg
import numba as numba
import numpy as np
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
from process_images import *
from pystripe.core import *
import matplotlib.pyplot as plt
def plot_images(img_list: List[ndarray], img_labels: List[str], vmax: int):
    fig, axes = plt.subplots(nrows=1, ncols=len(img_list), figsize=(20, 20))
    for idx, (im, label) in enumerate(zip(img_list, img_labels)):
        axes[idx].imshow(im, cmap='gray', vmin=0, vmax=vmax)
        axes[idx].set_title(label)
    plt.tight_layout()
    plt.show()

In [ ]:
from parallel_image_processor import *
tsv_volume = TSVVolume.load(r'E:\20230510_13_34_13_SM230308_05_LS_15x_800z_MIP_stitched\Ex_488_Em_525_MIP_xml_import_step_5.xml')
shape: Tuple[int, int, int] = tsv_volume.volume.shape  # shape is in z y x format
img = tsv_volume.imread(
    VExtent(
        tsv_volume.volume.x0, tsv_volume.volume.x1,
        tsv_volume.volume.y0, tsv_volume.volume.y1,
        tsv_volume.volume.z0 + shape[0]//2, tsv_volume.volume.z0 + shape[0]//2 + 1),
    tsv_volume.dtype)[0]
parallel_image_processor(
    source=TSVVolume.load(r'/data/20230419_17_34_03_SM221011_06_LS_15x_800z_stitched/Ex_488_Em_525_xml_import_step_5.xml'),
    destination=r"/data/20230419_17_34_03_SM221011_06_LS_15x_800z_stitched/Ex_488_Em_525_tif",
    fun=process_img,
    kwargs={'bleach_correction_frequency': 0.0005, 'bleach_correction_max_method': False, 'bleach_correction_y_slice_max': None, 'threshold': None, 'sigma': (4000.0, 4000.0), 'bidirectional': True, 'lightsheet': False, 'percentile': 0.25, 'rotate': 90, 'convert_to_8bit': False, 'bit_shift_to_right': 8, 'tile_size': (39220, 28056), 'd_type': 'uint16', "verbose": True},
    source_voxel=(0.8, 0.4, 0.4),
    target_voxel=20,
    max_processors=1
)

In [ ]:






img_paths = [
    Path("./tempData/cha1"),
    Path("./tempData/cha2"),
    Path("./tempData/cha3")
]
output_path = Path("./tempData/merged")
merge_all_channels(img_paths, output_path)

2023-08-29 21:29:36: using 8 workers. 791 images need to be processed.


RGB:  10%|#         | 83/791 [00:11<01:34,  7.47 images/s]


Terminating processes with dignity!


RGB:  12%|#1        | 94/791 [00:14<01:45,  6.63 images/s]

In [2]:
from cv2 import imread, IMREAD_ANYDEPTH, warpAffine, INTER_LINEAR, merge, WARP_INVERSE_MAP, MOTION_TRANSLATION, \
    findTransformECC, TERM_CRITERIA_COUNT, TERM_CRITERIA_EPS

def get_matrix_two_images(reference: ndarray,
                          subject: ndarray,
                          threshold: float = 90,
                          iterations: int = 10000,
                          termination: float = 1e-10):

    warp_matrix = eye(2, 3, dtype=float32)
    if reference is None or subject is None:
        return warp_matrix

    # generate matrices
    percentile_reference = prctl(reference, threshold)
    percentile_subject = prctl(subject, threshold)

    # scale images to max brightness at percentile; 0 <= pixel intensity <= 1
    reference = where(reference > percentile_reference, 1, reference / percentile_reference)
    subject = where(subject > percentile_subject, 1, subject / percentile_subject)

    criteria = (TERM_CRITERIA_EPS | TERM_CRITERIA_COUNT, iterations, termination)

    cc, warp_matrix = findTransformECC(reference, subject, warp_matrix, MOTION_TRANSLATION, criteria)
    return warp_matrix

In [5]:
image1 = imread("C:/Users/adnja/Documents/GitHub/image-preprocessing-pipeline/tempData/cha1/img_000279.tif", flags=IMREAD_ANYDEPTH)
image2 = imread("C:/Users/adnja/Documents/GitHub/image-preprocessing-pipeline/tempData/cha1/img_000279.tif", flags=IMREAD_ANYDEPTH)

print(get_matrix_two_images(image1, image2))


[[ 1.0000000e+00  0.0000000e+00 -4.7275461e-10]
 [ 0.0000000e+00  1.0000000e+00 -8.3534213e-10]]


In [6]:
from numpy import nonzero, array, sum, zeros, outer, linalg, transpose, diag, percentile, average, meshgrid, arange, dot, histogram
import pandas as pd  # used for writing points for file (testing only)

def get_points(stack: TifStack, threshold: list[float32]):
    """
    Parameters
    ----------
    stack: TifStack representing a single-channel 3D brain image
    threshold: range of percentile of pixels to be considered for points
               0 <= threshold[0], threshold[1] <= 100

    Returns
    -------
    points: list of tuples each containing three numbers representing a 3D point. (row, col, img)
    """
    points = []
    center_image = stack[stack.shape[0] // 2]
    lower_value = percentile(center_image, threshold[0])
    higher_value = percentile(center_image, threshold[1])
    for i in range(stack.shape[0]):
        row, col = nonzero((lower_value <= stack[i]) & (stack[i] <= higher_value))
        points += list(zip(row, col, [i] * stack.shape[0]))
    return points

# method from https://jingnanshi.com/blog/arun_method_for_3d_reg.html
def aruns_method_alignment(reference_input: ndarray, subject_input: ndarray):
    """
    Solve 3D registration using Arun's method: B = RA + t
    """

    # calculate centroids
    reference_centroid = array([sum(point[i] for point in reference_input) // len(reference_input) for i in range(3)])
    print("Reference centroid: ", end="")
    print(reference_centroid, end="")
    print(", " + str(len(reference_input)))
    subject_centroid = array([sum(point[i] for point in subject_input) // len(reference_input) for i in range(3)])
    print("Subject centroid: ", end=" ")
    print(subject_centroid, end="")
    print(", " + str(len(subject_input)))

    """
    # fix this section later
    n = min(len(reference_input), len(subject_input))
    reference = reference_input[:n]
    subject = subject_input[:n]

    # calculate the vectors from centroids
    reference_prime = reference - reference_centroid
    subject_prime = subject - subject_centroid

    # rotation estimation
    H = zeros([3, 3])
    for i in range(n):
        reference_i = reference_prime[:, i]
        subject_i = subject_prime[:, i]
        H = H + outer(reference_i, subject_i)
    U, S, V_transpose = linalg.svd(H)
    V = transpose(V_transpose)
    U_transpose = transpose(U)
    R = V @ diag([1, 1, linalg.det(V) * linalg.det(U_transpose)]) @ U_transpose

    # translation estimation
    t = subject_centroid - R @ reference_centroid

    return R, t
    """
    return 0, 0

def find_centroid(stack: TifStack, threshold: list[float32]):
    centroid = [0, 0, 0]
    total_weight = 0
    for i in range(stack.shape[0]):
        # print("image" + str(i))
        image = stack[i]

        # calculate thresholds
        min_brightness = percentile(image, threshold[0])
        max_brightness = percentile(image, threshold[1])

        # set all pixels below min and above max to 0
        image[image < min_brightness] = 0
        image[image > max_brightness] = 0

        # calculate weighted average
        x_coords, y_coords = meshgrid(arange(image.shape[1]), arange(image.shape[0]))
        centroid[0] += sum(x_coords * image)
        centroid[1] += sum(y_coords * image)
        image_weight = sum(image)
        centroid[2] += image_weight * i
        total_weight += image_weight

    return centroid / total_weight


img1 = TifStack(Path("./tempData/cha1_8b"))
img2 = TifStack(Path("./tempData/cha2_8b"))
img3 = TifStack(Path("./tempData/cha3_8b"))

img1_points = get_points(img1, [80, 80.01])
img2_points = get_points(img2, [80, 80.01])
img3_points = get_points(img3, [80, 80.01])

img1_centroid = find_centroid(img1, [50, 99.99])
print("Image 1 centroid: ", end="")
print(img1_centroid)

img2_centroid = find_centroid(img2, [50, 99.99])
print("Image 2 centroid: ", end="")
print(img2_centroid)

img3_centroid = find_centroid(img3, [50, 99.99])
print("Image 3 centroid: ", end="")
print(img3_centroid)

print("writing")
df1 = pd.DataFrame(img1_points)
df1.to_csv('./tempData/testing1.csv', index=False, header=False)
df2 = pd.DataFrame(img2_points)
df2.to_csv('./tempData/testing2.csv', index=False, header=False)
df3 = pd.DataFrame(img3_points)
df3.to_csv('./tempData/testing3.csv', index=False, header=False)
print("done")


Image 1 centroid: [1.38578978e-02 2.09442143e-02 4.01386303e+02]
Image 2 centroid: [-1.83427304e-02 -1.12064143e-04  3.85482661e+02]
Image 3 centroid: [4.38571846e-01 2.57226623e-01 4.06475204e+02]
writing
done


In [37]:
"""
Local Threshold: (exactly 114 points per image)
----------------
[80, 80.01]
Reference centroid: [542 553 386], 118905
Subject centroid:  [498 560 438], 125915

[99.79, 99.8]
Reference centroid: [562 646 394], 113905
Subject centroid: [581 653 386], 115208

[99.9, 100]
Reference centroid: [380 636 395], 625681
Subject centroid:  [390 630 381], 611524

[99.99, 100]
Reference centroid: [546 664 394], 113962
Subject centroid:  [898 984 678], 172199

Global Threshold: (Threshold estimated from center image, applied to all images)
-----------------
[80, 80.01]
Reference centroid: [579 561 387], 64984
Subject centroid:  [580 582 389], 66882

[99.79, 99.8]
Reference centroid: [536 661 430], 245306
Subject centroid:  [206 230 152], 87267

[99.9, 100]
Reference centroid: [282 600 376], 550598
Subject centroid:  [348 533 330], 442750

[99.99, 100]
Reference centroid: [507 587 288], 271767
Subject centroid:  [285 309 230], 125238
"""

'\nLocal Threshold: (exactly 114 points per image)\n----------------\n[80, 80.01]\nReference centroid: [542 553 386], 118905\nSubject centroid:  [498 560 438], 125915\n\n[99.79, 99.8]\nReference centroid: [562 646 394], 113905\nSubject centroid: [581 653 386], 115208\n\n[99.9, 100]\nReference centroid: [380 636 395], 625681\nSubject centroid:  [390 630 381], 611524\n\n[99.99, 100]\nReference centroid: [546 664 394], 113962\nSubject centroid:  [898 984 678], 172199\n\nGlobal Threshold: (Threshold estimated from center image, applied to all images)\n-----------------\n[80, 80.01]\n\n\n'

In [ ]:
# LOCAL
def get_points(stack: TifStack, threshold: list[float32]):
    """
    Parameters
    ----------
    stack: TifStack representing a single-channel 3D brain image
    threshold: range of percentile of pixels to be considered for points
               0 <= threshold[0], threshold[1] <= 100

    Returns
    -------
    points: list of tuples each containing three numbers representing a 3D point. (row, col, img)
    """
    points = []
    for i in range(stack.shape[0]):
        lower_value = percentile(stack[i], threshold[0])
        higher_value = percentile(stack[i], threshold[1])
        row, col = nonzero((lower_value <= stack[i]) & (stack[i] <= higher_value))
        points += list(zip(row, col, [i] * stack.shape[0]))
    return points

# GLOBAL
def get_points(stack: TifStack, threshold: list[float32]):
    """
    Parameters
    ----------
    stack: TifStack representing a single-channel 3D brain image
    threshold: range of percentile of pixels to be considered for points
               0 <= threshold[0], threshold[1] <= 100

    Returns
    -------
    points: list of tuples each containing three numbers representing a 3D point. (row, col, img)
    """
    points = []
    center_image = stack[stack.shape[0] // 2]
    lower_value = percentile(center_image, threshold[0])
    higher_value = percentile(center_image, threshold[1])
    for i in range(stack.shape[0]):
        row, col = nonzero((lower_value <= stack[i]) & (stack[i] <= higher_value))
        points += list(zip(row, col, [i] * stack.shape[0]))
    return points

In [5]:
from os import mkdir
paths = [Path("./tempData/cha1_8b"), Path("./tempData/cha2_8b")]
offset_arr = [-6, -7, -8, -9]

for p in offset_arr:
    offsets = [p]
    merge_path = Path("D:/BMAP_MergeData/cha1_2/testMerged_" + str(p))

    mkdir(merge_path)

    merge_all_channels(paths, offsets, merge_path)
    print("Merge on " + str(p) + " completed.")

print("process completed")

reference image index = 395
downsampling factor for transformation_matrix 1
[[ 1.0005251e+00 -8.3474268e-04 -1.5100256e-01]
 [ 2.0317000e-03  1.0049144e+00 -9.3056208e-01]
 [ 0.0000000e+00  0.0000000e+00  1.0000000e+00]]
2023-12-02 21:38:22: using 8 workers. 791 images need to be processed.


RGB: 100%|##########| 791/791 [00:19<00:00, 41.24 images/s]


Merge on -6 completed.
reference image index = 395
downsampling factor for transformation_matrix 1
[[ 1.000635   -0.00112438  0.01071943]
 [ 0.00222762  1.004591   -0.8861495 ]
 [ 0.          0.          1.        ]]
2023-12-02 21:45:20: using 8 workers. 791 images need to be processed.


RGB: 100%|##########| 791/791 [00:21<00:00, 37.35 images/s]


Merge on -7 completed.
reference image index = 395
downsampling factor for transformation_matrix 1
[[ 1.0009764  -0.00141568  0.02594887]
 [ 0.00293697  1.004254   -1.0330559 ]
 [ 0.          0.          1.        ]]
2023-12-02 21:52:13: using 8 workers. 791 images need to be processed.


RGB: 100%|##########| 791/791 [00:40<00:00, 19.56 images/s]


Merge on -8 completed.
reference image index = 395
downsampling factor for transformation_matrix 1
[[ 1.0010818  -0.00153864  0.05973725]
 [ 0.00342883  1.0039486  -1.0870677 ]
 [ 0.          0.          1.        ]]
2023-12-02 21:59:24: using 8 workers. 791 images need to be processed.


RGB: 100%|##########| 791/791 [00:40<00:00, 19.58 images/s]

Merge on -9 completed.
process completed
